In [1]:
import os

os.chdir("../utils")
print(os.getcwd())

/home/dell/dell/ralph projects/ai_agent_for_advertisement/backend/utils


In [2]:
from llm_call import GeminiService
service = GeminiService()

/home/dell/miniconda3/envs/client_accelerator/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/dell/dell/ralph projects/ai_agent_for_advertisement/backend/utils/llm_call.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [6]:
from liteparse import LiteParse

parser = LiteParse()
result = parser.parse("/home/dell/dell/ralph projects/ai_agent_for_advertisement/backend/test/reference_pack.pdf")

[liteparse] extract: 147.7ms (17 pages)
[liteparse] ocr render: 58.3ms (1 pages)
[liteparse] ocr: 761.6ms
[liteparse] project: 15.0ms
[liteparse] total: 983.1ms


In [9]:
# 1. Read the system prompt from a text file
file_path = "/home/dell/dell/ralph projects/ai_agent_for_advertisement/backend/prompts/reference_pack_extraction.txt"

with open(file_path, "r", encoding="utf-8") as file:
    my_system_instruction = file.read().strip()

# 2. Define the user prompt as a string
my_user_prompt = result.text
# 3. Initialize the service (assumes GeminiService is defined in a previous cell or imported)
gemini_service = GeminiService(model_name="gemini-2.5-flash")

print("Generating response...\n" + "-"*40)

# 4. Generate the text by passing both arguments
try:
    response = gemini_service.generate_text(
        prompt=my_user_prompt,
        system_instruction=my_system_instruction,
        temperature=0 # Adjusted slightly for a more focused answer
    )
    print(response)
except Exception as e:
    print(f"Failed to generate text: {e}")

Generating response...
----------------------------------------
```json
{
  "clients": [
    {
      "id": "a1b2c3d4-e5f6-4a7b-8c9d-0e1f2a3b4c5d",
      "name": "Marcus Vane",
      "brand": "The Sovereign Storefront",
      "tone": "Plainspoken, unhurried, anti-hype, operationally specific",
      "voice_description": "Talks like a warehouse manager explaining how something actually works, not a salesman pushing a dream. Leans into skepticism, explains the mechanism, and lets the structure sell. Uses specific language, positions himself as an operator not a guru, is honest about effort, and remains calm without urgency theatre.",
      "market_stage": "scaling",
      "core_emotion": "Earned Calm"
    }
  ],
  "offers": [
    {
      "id": "b1c2d3e4-f5a6-4b7c-8d9e-1f2a3b4c5d6e",
      "client_id": "a1b2c3d4-e5f6-4a7b-8c9d-0e1f2a3b4c5d",
      "name": "First Pallet trial workshop",
      "type": "front_end",
      "price": 1.00,
      "description": "A $1 two-hour live workshop that wa

In [11]:
import json
import re
import uuid
import psycopg2
import psycopg2.extras
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

# --- 1. Parse JSON — strip markdown fences Gemini may add ---
raw = response.strip()
match = re.search(r"```(?:json)?\s*([\s\S]*?)```", raw)
if match:
    raw = match.group(1).strip()
data = json.loads(raw)

# --- 2. Remap UUIDs — Gemini produces non-hex chars (g–z) which PostgreSQL rejects ---
# Build a mapping from every Gemini-generated id → a valid uuid4
id_remap = {}
for table_rows in data.values():
    for row in table_rows:
        old_id = row.get("id")
        if old_id and old_id not in id_remap:
            id_remap[old_id] = str(uuid.uuid4())

def remap_row(row):
    # Replace any value (id OR foreign key) that appears in the remap dict
    return {
        k: id_remap[v] if isinstance(v, str) and v in id_remap else v
        for k, v in row.items()
    }

for table in data:
    data[table] = [remap_row(row) for row in data[table]]

# --- 3. Insert in FK-safe order ---
INSERTION_ORDER = [
    "clients",            # no FK deps
    "offers",             # → clients
    "avatars",            # → clients
    "mechanisms",         # → clients
    "proof_bank",         # → clients
    "angles",             # → clients
    "hooks",              # → clients
    "script_frameworks",  # no FK deps
    "constraints",        # → clients
    "pain_points",        # → avatars
    "ideas",              # → clients, angles
]

JSONB_COLS = {"structure"}  # columns that need to be serialised as JSONB

conn = psycopg2.connect(
    host = os.getenv("DB_HOST", "localhost"),
    port = os.getenv("POSTGRES_PORT", "5432"),
    dbname = os.getenv("POSTGRES_DB", "app_db"),
    user = os.getenv("POSTGRES_USER", "app_user"),
    password = os.getenv("POSTGRES_PASSWORD", ""),
)

try:
    with conn:
        with conn.cursor() as cur:
            for table in INSERTION_ORDER:
                rows = data.get(table, [])
                if not rows:
                    print(f"  {table}: skipped (empty)")
                    continue

                columns  = list(rows[0].keys())
                col_str  = ", ".join(columns)
                ph_str   = ", ".join(["%s"] * len(columns))
                sql = (
                    f"INSERT INTO {table} ({col_str}) "
                    f"VALUES ({ph_str}) "
                    f"ON CONFLICT (id) DO NOTHING"
                )

                values = [
                    tuple(
                        psycopg2.extras.Json(row[col])
                        if col in JSONB_COLS and isinstance(row[col], (dict, list))
                        else row[col]
                        for col in columns
                    )
                    for row in rows
                ]

                cur.executemany(sql, values)
                print(f"  {table}: {len(rows)} row(s) inserted")

    print("\nDone — all tables saved.")
except Exception as e:
    print(f"\nFailed: {e}")
    raise
finally:
    conn.close()


  clients: 1 row(s) inserted
  offers: 5 row(s) inserted
  avatars: 5 row(s) inserted
  mechanisms: 2 row(s) inserted
  proof_bank: 11 row(s) inserted
  angles: 4 row(s) inserted
  hooks: 13 row(s) inserted
  script_frameworks: 1 row(s) inserted
  constraints: 8 row(s) inserted
  pain_points: 11 row(s) inserted
  ideas: skipped (empty)

Done — all tables saved.
